# Step 2 · PINN — learn ONE solution from physics

A physics-informed neural network $u_\theta(x,t)$ trained with **no data** — only the
PDE residual $u_t + u u_x - \nu u_{xx}$, the initial condition, and a soft periodic
boundary condition. This notebook is a thin wrapper over the `shocklab` package:
it loads the trained checkpoint and reproduces the evaluation against the exact
Cole–Hopf solution. To (re)train: `python -m shocklab.train.pinn --config configs/pinn_classic.yaml`.

In [ ]:
import json
from pathlib import Path

import torch
import yaml

from shocklab.models.pinn import PINN
from shocklab.train.common import get_device
from shocklab.train.pinn import evaluate

ckpt = Path("checkpoints/pinn_classic")
cfg = yaml.safe_load((ckpt / "config.yaml").read_text())
device = get_device("cuda")

model = PINN(cfg["n_hidden_layers"], cfg["width"]).to(device)
model.load_state_dict(torch.load(ckpt / "model.pt", map_location=device))
model.eval()
print(f"PINN: {model.num_params()} params ({cfg['n_hidden_layers']}x{cfg['width']} tanh MLP)")

In [ ]:
# reproduce the evaluation vs the exact (Cole–Hopf) solution on the 1024x101 grid
res = evaluate(model, device)
for k, v in res.items():
    print(f"{k:10s} {v:.3e}" if isinstance(v, float) else f"{k:10s} {v}")

# and the numbers recorded at training time
print("\nstored metrics.json:")
print(json.dumps(json.loads((ckpt / "metrics.json").read_text())["results"], indent=2))

In [ ]:
from IPython.display import Image

# the error concentrates at the shock — the whole point of the PINN demo
Image(filename="checkpoints/pinn_classic/figures/field_comparison.png")

**Takeaway.** The PINN reaches rel L2 ≈ 5e-4 on the classic case — razor-sharp —
but it learned exactly *one* solution and took ~20 min. Step 3 learns the whole
solution *operator* instead.